# 上下文工程
---
[上下文工程](https://docs.langchain.com/oss/python/langchain/context-engineering)（Context Engineering）对于 Agent 得出正确的结果至关重要。模型回答不好，很多时候不是因为能力不足，而是因为没有获得足以推断出正确结果的上下文信息。通过上下文工程，增强 Agent 获取和管理上下文的能力，是很有必要的。

**LangGraph 将上下文分为三种类型：**

- 模型上下文（Model Context）（瞬态）：每次调用的输入
- 工具上下文（Tool Context）（持久化）：工具的读写
- 生命周期上下文（Life-cycle Context）（持久化）：调用之间的处理

**三种数据源**

- Runtime Context：静态配置（用户 ID、API 密钥）
- State：短期记忆（消息历史、临时数据）
- Store：长期记忆（用户偏好、历史见解）

无论哪种 Context，都需要定义它的 Schema。在这方面，LangGraph 提供了相当高的自由度，你可以使用 `dataclasses`、`pydantic`、`TypedDict` 这些包的任意一个创建你的 Context Schema.

In [34]:
import os
import uuid
import sqlite3

from typing import Callable
from dotenv import load_dotenv
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, wrap_model_call, ModelRequest, ModelResponse, SummarizationMiddleware
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.sqlite import SqliteStore

# 加载模型配置
_ = load_dotenv()

# 加载模型
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model="qwen3-max",#os.getenv("MODEL_ID"),
    temperature=0.7,
)

## 一、动态修改系统提示词/工具

上下文工程与前序章节的中间件（middleware）和记忆（memory）密不可分。上下文的具体实现依赖中间件，而上下文的存储则依赖记忆系统。具体来讲，LangGraph 预置了 `@dynamic_prompt` 中间件，用于动态修改系统提示词。

既然是动态修改，肯定需要某个条件来触发修改。除了开发触发逻辑，我们还需要从智能体中获取触发逻辑所需的即时变量。这些变量通常存储在以下三个存储介质中：

- 运行时（Runtime）- 所有节点共享一个 Runtime。同一时刻，所有节点取到的 Runtime 的值是相同的。一般用于存储时效性要求较高的信息。
- 短期记忆（State）- 在节点之间按顺序传递，每个节点接收上一个节点处理后的 State。主要用于存储 Prompt 和 AI Message。
- 长期记忆（Store）- 负责持久化存储，可以跨 Workflow / Agent 保存信息。可以用来存用户偏好、以前算过的统计值等。

以下三个例子，分别演示如何使用来自 Runtime、State、Store 中的上下文，编写触发条件。

### 1）使用 `State` 管理上下文

利用 `State` 中蕴含的信息操纵 system prompt.

In [37]:
@dynamic_prompt
def state_aware_prompt(request: ModelRequest) -> str:
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    base = "You are a helpful assistant."

    if message_count > 6:
        base += "\nThis is a long conversation - be extra concise."

    # 临时打印base看效果
    print(base)

    return base

agent = create_agent(
    model=llm,
    middleware=[state_aware_prompt]
)

result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "广州今天的天气怎么样？"},
        {"role": "assistant", "content": "广州天气很好"},
        {"role": "user", "content": "吃点什么好呢"},
        {"role": "assistant", "content": "要不要吃香茅鳗鱼煲"},
        {"role": "user", "content": "香茅是什么"},
        {"role": "assistant", "content": "香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉"},
        {"role": "user", "content": "auv 那还等什么，咱吃去吧"},
    ]},
)

for message in result['messages']:
    message.pretty_print()

You are a helpful assistant.
This is a long conversation - be extra concise.
================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================

广州天气很好
================================ Human Message =================================

吃点什么好呢
================================== Ai Message ==================================

要不要吃香茅鳗鱼煲
================================ Human Message =================================

香茅是什么
================================== Ai Message ==================================

香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉
================================ Human Message =================================

auv 那还等什么，咱吃去吧
================================== Ai Message ==================================

走起！香茅鳗鱼煲安排上～ 🍲


把 `message_count > 6` 里的 6 改成 7，试试看会发生什么。

### 2）使用 `Store` 管理上下文

In [38]:
@dataclass
class Context:
    user_id: str

@dynamic_prompt
def store_aware_prompt(request: ModelRequest) -> str:
    user_id = request.runtime.context.user_id

    # Read from Store: get user preferences
    store = request.runtime.store
    user_prefs = store.get(("preferences",), user_id)

    base = "You are a helpful assistant."

    if user_prefs:
        style = user_prefs.value.get("language", "Chinese")
        base += f"\nUser prefers {style} responses."

    return base

store = InMemoryStore()

agent = create_agent(
    model=llm,
    middleware=[store_aware_prompt],
    context_schema=Context,
    store=store,
)

# 预置两条偏好信息
store.put(("preferences",), "user_1", {"language": "Chinese"})
store.put(("preferences",), "user_2", {"language": "Korean"})

In [39]:
# 用户1喜欢中文回复
result = agent.invoke(
    {"messages": [
        {"role": "system", "content": "You are a helpful assistant. Please be extra concise."},
        {"role": "user", "content": '无脊柱动物有哪些？'}
    ]},
    context=Context(user_id="user_1"),
)

for message in result['messages']:
    message.pretty_print()

================================ System Message ================================

You are a helpful assistant. Please be extra concise.
================================ Human Message =================================

无脊柱动物有哪些？
================================== Ai Message ==================================

无脊椎动物包括昆虫、蜘蛛、螃蟹、虾、蜗牛、章鱼、水母、蚯蚓、海星等，种类繁多，占动物界95%以上。


In [40]:
# 用户2喜欢韩文回复
result = agent.invoke(
    {"messages": [
        {"role": "system", "content": "You are a helpful assistant. Please be extra concise."},
        {"role": "user", "content": '无脊柱动物有哪些？'}
    ]},
    context=Context(user_id="user_2"),
)

for message in result['messages']:
    message.pretty_print()

================================ System Message ================================

You are a helpful assistant. Please be extra concise.
================================ Human Message =================================

无脊柱动物有哪些？
================================== Ai Message ==================================

무척추동물에는 곤충, 거미, 갑각류, 연체동물(오징어, 달팽이 등), 해파리, 지렁이, 성게 등이 있습니다.


### 3）使用 `Runtime` 管理上下文

In [50]:
@dataclass
class Context:
    user_name: str
    user_role: str
    language: str = "zh"

@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    """根据用户信息生成个性化提示词"""
    ctx = request.runtime.context

    # 基础提示词
    base = f"你是一个专业的助手。用户名是 {ctx.user_name}。"

    # 根据角色添加指令
    if ctx.user_role == "admin":
        base += "\n用户是管理员，可以执行所有操作。"
    elif ctx.user_role == "guest":
        base += "\n用户是访客，只能进行查询操作。"
    else:
        base += "\n用户是普通用户。"

    # 根据语言调整
    if ctx.language == "en":
        base += "\nPlease respond in English."
    elif ctx.language == "ko":
        base += "\nPlease respond in Korean."
    else:
        base += "\n请用中文回复。"
    print("DEBUG: ", base, ctx.language)

    return base

In [54]:
agent = create_agent(
    model=llm,
    middleware=[personalized_prompt],
    context_schema=Context,
    store = InMemoryStore()
)

# 使用
result = agent.invoke(
    {"messages": [{"role": "user", "content": "帮我查看订单"}]},
    context=Context(user_name="dingshixiong", user_role="guest", language="zh"),
)
for message in result['messages']:
    message.pretty_print()

DEBUG:  你是一个专业的助手。用户名是 dingshixiong。
用户是访客，只能进行查询操作。
请用中文回复。 zh
================================ Human Message =================================

帮我查看订单
================================== Ai Message ==================================

您好，dingshixiong！  
由于您当前是访客身份，系统权限仅支持查询操作。要查看您的订单信息，请先登录您的账户。

如果您已有账号，请登录后再次尝试查看订单；如果尚未注册，建议您完成注册以便管理您的订单和相关信息。

如有其他问题，也欢迎随时告诉我！


## 二、动态修改消息列表

LangGraph 预制了动态修改消息列表（Messages）的中间件 `@wrap_model_call`。上一节已经演示如何从 `State`、`Store`、`Runtime` 中获取上下文，本节将不再一一演示。在下面这个例子中，我们主要演示如何使用 `Runtime` 将本地文件的内容注入消息列表。

In [32]:
# !pip install ipynbname

In [55]:
@dataclass
class FileContext:
    uploaded_files: list[dict]

@wrap_model_call
def inject_file_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """注入有关用户在此会话中上传的文件的上下文信息."""
    uploaded_files = request.runtime.context.uploaded_files

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except Exception as e:
        import ipynbname
        import os
        notebook_path = ipynbname.path()
        base_dir = os.path.dirname(notebook_path)

    file_sections = []
    for file in uploaded_files:
        name, ftype = "", ""
        path = file.get("path")
        if path:
            base_filename = os.path.basename(path)
            stem, ext = os.path.splitext(base_filename)
            name = stem or base_filename
            ftype = (ext.lstrip(".") if ext else None)

            # 构建文件描述内容
            content_list = [f"名称: {name}"]
            if ftype:
                content_list.append(f"类型: {ftype}")

            # 解析相对路径为绝对路径
            abs_path = path if os.path.isabs(path) else os.path.join(base_dir, path)

            # 读取文件内容
            content_block = ""
            if abs_path and os.path.exists(abs_path):
                try:
                    with open(abs_path, "r", encoding="utf-8") as f:
                        content_block = f.read()
                except Exception as e:
                    content_block = f"[读取文件错误 '{abs_path}': {e}]"
            else:
                content_block = "[文件路径缺失或未找到]"

            section = (
                f"---\n"
                f"{chr(10).join(content_list)}\n\n"
                f"{content_block}\n"
                f"---"
            )
            file_sections.append(section)

        file_context = (
            "已加载的会话文件：\n"
            f"{chr(10).join(file_sections)}"
            "\n回答问题时请参考这些文件。"
        )

        # 在最近的消息之前注入文件上下文
        messages = [  
            *request.messages,
            {"role": "user", "content": file_context},
        ]
        request = request.override(messages=messages)  

    return handler(request)

agent = create_agent(
    model=llm,
    middleware=[inject_file_context],
    context_schema=FileContext,
)

In [56]:
result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "关于上海地铁的无脸乘客，有什么需要注意的？",
        }],
    },
    context=FileContext(uploaded_files=[{"path": "./docs/rule_horror.md"}]),
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

关于上海地铁的无脸乘客，有什么需要注意的？
================================== Ai Message ==================================

关于“上海地铁的无脸乘客”，根据你提供的《上海规则怪谈》中的第二条规则，有以下几点**必须严格注意**：

---

### 🚇 **关键规则：地铁上的陌生人**
> 地铁运营结束后，若你仍身处车厢内，会出现一位无脸乘客。他会低声问：“你要去哪？”  
> **只能报一个真实存在的上海地名。**  
> 说出不存在的地名或保持沉默，你会在车厢内看到自己的尸体。

---

### ✅ 正确应对方式：
1. **确认地铁是否已结束运营**：上海地铁通常在23:00–24:00之间结束当日运营（具体线路略有不同）。若你在末班车之后仍留在车厢内，即进入“异常状态”。
2. **若遇到无脸乘客**（面部模糊、空白或被阴影覆盖）：
   - 他低声问：“你要去哪？”
   - **必须立刻、清晰地回答一个真实存在的上海地名**，例如：
     - “人民广场”
     - “徐家汇”
     - “陆家嘴”
     - “静安寺”
     - “五角场”
   - 地名必须是**官方存在、可查证的地铁站或行政区**，不可虚构（如“幻梦路”“永夜街”等）。
3. **切勿沉默、犹豫或编造地名**：后果极其严重——你会当场看到自己的尸体，暗示灵魂已被剥离或取代。

---

### ⚠️ 额外提醒：
- 最安全的做法是**绝不滞留于末班地铁之后的车厢内**。一旦发现列车已过末班时间仍在运行，应立即在下一站下车。
- 若车厢内灯光忽明忽暗、其他乘客全部消失、广播无声，即为“异常空间”开启的征兆，需高度警惕。
- 回答地名后，无论无脸乘客有何反应（如点头、消失、继续凝视），**不要追问、不要回头、不要再次搭话**，安静等待列车到站后迅速离开。

---

记住：**在上海的深夜，规则就是生存的边界。**  
违者——后果自负。


## 三、在工具中使用上下文

下面，我们尝试在工具中使用存储在 `SqliteStore` 中的上下文信息。

In [57]:
# 删除SQLite数据库
if os.path.exists("db/user-info.db"):
    os.remove("db/user-info.db")

# 创建SQLite存储
conn = sqlite3.connect("db/user-info.db", check_same_thread=False, isolation_level=None)
conn.execute("PRAGMA journal_mode=WAL;")
conn.execute("PRAGMA busy_timeout = 30000;")

store = SqliteStore(conn)

# 预置两条用户信息
store.put(("user_info",), "柳如烟", {"description": "清冷才女，身怀绝技，为寻身世之谜踏入江湖。", "birthplace": "吴兴县"})
store.put(("user_info",), "苏慕白", {"description": "孤傲剑客，剑法超群，背负家族血仇，隐于市井追寻真相。", "birthplace": "杭县"})

### 1）基础用例

使用 `ToolRuntime`

In [58]:
@tool
def fetch_user_data(
    user_id: str,
    runtime: ToolRuntime
) -> str:
    """
    从内存存储中获取用户信息。
    
    :param user_id: 用户的唯一标识符
    :param runtime: 框架注入的工具运行时上下文
    :return: 如果找到用户描述字符串，则返回该字符串；否则返回空字符串
    """
    store = runtime.store
    user_info = store.get(("user_info",), user_id)

    user_desc = ""
    if user_info:
        user_desc = user_info.value.get("description", "")

    return user_desc

agent = create_agent(
    model=llm,
    tools=[fetch_user_data],
    store=store,
)

In [59]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "我要柳如烟的全部信息"
    }]
})

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

我要柳如烟的全部信息
================================== Ai Message ==================================
Tool Calls:
  fetch_user_data (call_44cbff142729495aafbd5e4d)
 Call ID: call_44cbff142729495aafbd5e4d
  Args:
    user_id: 柳如烟
================================= Tool Message =================================
Name: fetch_user_data

清冷才女，身怀绝技，为寻身世之谜踏入江湖。
================================== Ai Message ==================================

柳如烟是一位清冷才女，身怀绝技，为探寻自身身世之谜而踏入江湖。


### 2）复杂一点的例子

使用 `ToolRuntime[Context]`

In [60]:
@dataclass
class Context:
    key: str

@tool
def fetch_user_data(
    user_id: str,
    runtime: ToolRuntime[Context]
) -> str:
    """
    从内存存储中获取用户信息。
    
    :param user_id: 用户的唯一标识符
    :param runtime: 框架注入的工具运行时上下文
    :return: 如果找到用户描述字符串，则返回该字符串；否则返回空字符串
    """
    key = runtime.context.key

    store = runtime.store
    user_info = store.get(("user_info",), user_id)

    user_desc = ""
    if user_info:
        user_desc = user_info.value.get(key, "")

    return f"{key}: {user_desc}"

agent = create_agent(
    model=llm,
    tools=[fetch_user_data],
    store=store,
)

In [61]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "我要柳如烟的全部信息"}]},
    context=Context(key="birthplace"),
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

我要柳如烟的全部信息
================================== Ai Message ==================================
Tool Calls:
  fetch_user_data (call_72602537b095458cbab020fd)
 Call ID: call_72602537b095458cbab020fd
  Args:
    user_id: 柳如烟
================================= Tool Message =================================
Name: fetch_user_data

birthplace: 吴兴县
================================== Ai Message ==================================

柳如烟的全部信息目前包括：出生地为吴兴县。如果需要更多详细信息，请提供进一步的说明或补充数据。


### 四、压缩上下文

LangChain 提供了内置的中间件 `SummarizationMiddleware` 用于压缩上下文。该中间件维护的是典型的 **生命周期上下文**，与 **模型上下文** 和 **工具上下文** 的瞬态更新不同，生命周期上下文会持续更新：持续将旧消息替换为摘要。

除非上下文超长，导致模型能力降低，否则不需要使用 `SummarizationMiddleware`。一般来说，触发摘要得值可以设得较大。比如：

- `max_tokens_before_summary`: 4000
- `messages_to_keep`: 20

In [74]:
# 创建短期记忆
checkpointer = InMemorySaver()

# 创建带内置摘要中间件的Agent
# 为了让配置能在我们的例子里生效，这里的触发值设得很小
agent = create_agent(
    model=llm,
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=('tokens', 40),  # Trigger summarization at 40 tokens
            keep=('messages', 1),  # Keep last 1 messages after summary
        ),
    ],
)

In [75]:
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "广州今天的天气怎么样？"},
        {"role": "assistant", "content": "广州天气很好"},
        {"role": "user", "content": "吃点什么好呢"},
        {"role": "assistant", "content": "要不要吃香茅鳗鱼煲"},
        {"role": "user", "content": "香茅是什么"},
        {"role": "assistant", "content": "香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉"},
        {"role": "user", "content": "auv 那还等什么，咱吃去吧"},
    ]},
    checkpointer=checkpointer,
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户提供了一段对话，但没有明确的整体目标或请求。这段对话主要围绕广州的天气和食物选择展开。

## SUMMARY
- 用户询问了广州今天的天气情况，回复是天气很好。
- 用户接着询问吃什么好，建议尝试香茅鳗鱼煲。
- 用户对香茅不熟悉，解释说香茅又名柠檬草，在泰式冬阴功汤、越南烤肉中常见。

## ARTIFACTS
None

## NEXT STEPS
None
================================ Human Message =================================

auv 那还等什么，咱吃去吧
================================== Ai Message ==================================

听起来您已经准备好去享受美食了！香茅鳗鱼煲确实是一个不错的选择，尤其是在这样好天气的日子里。如果您还有其他想尝试的菜品或者需要推荐附近的餐厅，请告诉我，我很乐意为您提供更多信息。祝您用餐愉快！如果需要帮助找地方或者有其他问题，随时欢迎询问哦。
